# 馬券風 次単語予想モード — レースデータの作成LLM が次の単語を決めるまでを競馬に見立てるための CSV を作ります。- **出走馬** = 次の単語の候補- **着順** = 最終出力確率の順位- **途中経過** = 各層で logit lens を掛けた確率（1層 = 100m）出力した `races.csv` をアプリの設定画面「コンテンツ」→ 馬券風 次単語予想 で取り込み、そのあとオッズ平均・分散を GUI で調整します。**このノートブックで初期値まで入れる**ので、そのままでも遊べます。Colab のランタイムは **GPU 不要**です（150m〜1.8b 程度なら CPU で動きます）。

## 1. 準備

In [ ]:
!pip -q install "transformers>=4.44" "torch>=2.3" accelerate

In [ ]:
import math, re, jsonimport numpy as npimport pandas as pdimport torchimport matplotlib.pyplot as pltfrom transformers import AutoModelForCausalLM, AutoTokenizer# 日本語が出せるモデルなら何でもよい。層が多いほどレースが長くなるMODEL = "llm-jp/llm-jp-3-1.8b"   # 24層 → 2400mN_HORSES = 12                     # 出走頭数（8〜18）METERS_PER_LAYER = 100            # アプリ側の既定と合わせるtok = AutoTokenizer.from_pretrained(MODEL)model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)model.eval()n_layers = model.config.num_hidden_layersprint(f"{MODEL}  {n_layers}層 → {n_layers * METERS_PER_LAYER}m")

## 2. logit lens各層の隠れ状態を**最終層の LayerNorm と出力埋め込みに通して**、その層の時点での「次の単語の確率」を見ます。層 0 は入力そのもので意味がないので 1 層目から使います。

In [ ]:
@torch.no_grad()def layer_probs(prompt: str, k: int = N_HORSES):    """最終確率の上位 k 語について、各層での確率を返す。    戻り値: (語のリスト, 最終確率 [k], 層ごとの確率 [k, 層数])    """    ids = tok(prompt, return_tensors="pt")    out = model(**ids, output_hidden_states=True)    # 最終出力（着順の根拠）    final = torch.softmax(out.logits[0, -1].float(), dim=-1)    top = torch.topk(final, k)    idx = top.indices    words = [tok.decode([i]) for i in idx.tolist()]    # 各層を出力空間へ射影する。norm の名前はモデルによって違うので拾いに行く    base = model.model if hasattr(model, "model") else model.transformer    norm = getattr(base, "norm", None) or getattr(base, "ln_f", None)    head = model.get_output_embeddings()    per_layer = []    for h in out.hidden_states[1:]:            # 1層目から        z = h[0, -1].float()        if norm is not None:            z = norm(z.to(next(norm.parameters()).dtype)).float()        logits = head(z.to(head.weight.dtype)).float()        p = torch.softmax(logits, dim=-1)[idx]        per_layer.append(p.numpy())    return words, top.values.numpy(), np.stack(per_layer, axis=1)   # [k, 層数]

## 3. 順位変動を見る**縦軸 logit** と**縦軸 probability** の 2 枚を出します。順位がコロコロ入れ替わりすぎるレースはここで気づけるので、プロンプトを変えるか、頭数を減らす判断ができます。アプリ側でも平滑化は掛かりますが、**元データが暴れすぎているとレースになりません**。

In [ ]:
def plot_race(words, probs, title=""):    """probs: [頭, 層]"""    T = probs.shape[1]    x = np.arange(1, T + 1) * METERS_PER_LAYER    fig, axes = plt.subplots(1, 2, figsize=(15, 5))    for ax, (vals, name) in zip(axes, [(np.log(probs + 1e-12), "logit (log prob)"), (probs, "probability")]):        for i, w in enumerate(words):            ax.plot(x, vals[i], label=w, linewidth=1.6)        ax.set_xlabel("距離 [m]（1層 = %dm）" % METERS_PER_LAYER)        ax.set_ylabel(name)        ax.grid(alpha=.3)    axes[1].legend(fontsize=8, ncol=2, loc="upper left")    fig.suptitle(title)    plt.tight_layout()    plt.show()

## 4. オッズの初期値手で 1 語ずつ入れるのは大変なので、ここで初期値を作ります。```oddsMean = clamp(0.80 / 最終確率, 1.1, 500)    ← 控除率 20% ぶんだけ甘いoddsVar  = (0.18 × oddsMean)²                  ← 変動係数 18%```**上位人気が実際に来やすい**設定です。荒れさせたいレースだけ、アプリの GUI で分散を上げてください。

In [ ]:
TAKEOUT = 0.80      # 単勝の控除率CV = 0.18           # オッズの変動係数（大きいほど荒れる）def initial_odds(p):    mean = float(np.clip(TAKEOUT / max(p, 1e-4), 1.1, 500))    return mean, (CV * mean) ** 2

## 5. レースを作るプロンプトを並べて実行します。5 レースぶんが既定です。

In [ ]:
PROMPTS = [    ("R1", "今日は"),    ("R2", "大学の研究室で"),    ("R3", "人工知能は"),    ("R4", "東京の天気は"),    ("R5", "この問題の答えは"),]rows = []for race_id, prompt in PROMPTS:    words, final, per_layer = layer_probs(prompt)    plot_race(words, per_layer, f"{race_id}  「{prompt}」")    for i, w in enumerate(words):        mean, var = initial_odds(float(final[i]))        row = {            "race_id": race_id,            "race_name": f"第{race_id[1:]}R 「{prompt}」",            "prompt": prompt,            "model": MODEL,            "word": w,            "final_prob": float(final[i]),            "odds_mean": round(mean, 1),            "odds_var": round(var, 2),        }        for t in range(per_layer.shape[1]):            row[f"layer_{t+1}"] = float(per_layer[i, t])        rows.append(row)df = pd.DataFrame(rows)print(df.shape)df.head(12)

## 6. 書き出し`races.csv` をダウンロードして、アプリの設定画面から取り込みます。

In [ ]:
df.to_csv("races.csv", index=False, encoding="utf-8-sig")try:    from google.colab import files    files.download("races.csv")except Exception:    print("races.csv を書き出しました。")

## 補足：荒れ具合の調整| やりたいこと | 触るところ || --- | --- || 人気どおりに決まりやすくする | `CV` を下げる（0.10 など） || 荒れさせる | `CV` を上げる（0.30 など）。または GUI で特定の語の分散だけ上げる || レースを長くする | 層の多いモデルにする。`METERS_PER_LAYER` は見た目の距離が変わるだけ || 途中の入れ替わりを増やす | プロンプトを曖昧なものにする（続きが一意に決まらないほど暴れる） |着順は**最終出力確率だけ**で決まります。途中経過をどういじっても結果は変わりません。